# Phase 2: Legal Document Preprocessing and Sentence-Aware Chunking

**Project**: Enhanced Legal Document Similarity and Case Retrieval Using Hybrid BERT, Legal Knowledge, and Ensemble Learning  
**Dataset**: `Exploration-Lab/IL-TUR` (Prior Case Retrieval - PCR)  

This notebook demonstrates the reproducible preprocessing pipeline designed for Indian legal jurisprudence:
1. **Text Normalization**: Unicode NFKC, whitespace standardization, control character removal while strictly preserving citations, case numbers, and statutory sections.
2. **Document Reconstruction**: Safe assembly of sentence lists into cohesive judgments without inventing missing text.
3. **Sentence-Aware Chunking**: Token-bounded transformer chunking with configurable overlap, preserving sentence boundaries and metadata.
4. **Retrieval Integrity**: Partition isolation, query-candidate alignment, and zero-relevant candidate tracking.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing.text_cleaner import LegalTextCleaner
from src.preprocessing.document_reconstructor import DocumentReconstructor
from src.preprocessing.chunker import SentenceAwareChunker

print("Modules successfully loaded!")

## 1. Text Normalization and Citation Preservation

Legal text contains domain-specific entities (e.g., `Section 302 IPC`, `AIR 1965 SC 123`, `(2008) 4 SCC 594`) that must remain pristine during cleaning.

In [ ]:
cleaner = LegalTextCleaner()

sample_raw = (
    "The\u00a0appellant was convicted under\u200b Section 302 read with Section 34 of the Indian Penal Code, 1860.\r\n"
    "Reliance was placed on AIR 1965 SC 123 and (2008) 4 SCC 594.\x00\x08   The appeal is dismissed."
)

cleaned = cleaner.clean(sample_raw)
print("--- RAW INPUT ---")
print(repr(sample_raw))
print("\n--- CLEANED OUTPUT ---")
print(cleaned)

## 2. Document Reconstruction

Reconstructs full text from sentence arrays while filtering nulls and empty entries.

In [ ]:
reconstructor = DocumentReconstructor(cleaner=cleaner)

raw_sentences = [
    "This criminal appeal arises out of the judgment passed by the High Court.",
    None,
    "   ",
    "The prosecution alleged that the accused committed offenses punishable under Section 498-A IPC.",
    "The trial court acquitted the respondent due to lack of corroborating evidence."
]

result = reconstructor.reconstruct(raw_sentences)
print("Sentence Count:", result["sentence_count"])
print("Empty Sentences Removed:", result["empty_sentences_count"])
print("Word Count:", result["word_count"])
print("\nReconstructed Full Text:")
print(result["full_text"])

## 3. Sentence-Aware Chunking for Transformers

Chunks documents into token-bounded spans (default max 512 tokens with 64 token overlap) without cutting sentences in half.

In [ ]:
chunker = SentenceAwareChunker(
    tokenizer=project_root / "models" / "bert" / "tokenizer.json",
    max_tokens=512,
    overlap_tokens=64
)

# Load a sample case from raw external dataset
sample_parquet = project_root / "data" / "external" / "IL-TUR-PCR" / "dev_queries" / "dev_queries.parquet"
df_sample = pq.read_table(sample_parquet).to_pandas()
sample_row = df_sample.iloc[0]

recon_sample = reconstructor.reconstruct(sample_row["text"])
chunks = chunker.chunk_document(doc_id=str(sample_row["id"]), sentences=recon_sample["clean_sentences"])

print(f"Document ID: {sample_row['id']}")
print(f"Total Clean Sentences: {recon_sample['sentence_count']}")
print(f"Total Generated Chunks: {len(chunks)}")
print("\nSample First Chunk:")
for k, v in chunks[0].items():
    if k == 'text':
        print(f"  {k}: {v[:120]}...")
    else:
        print(f"  {k}: {v}")

## 4. Preprocessing Dataset Inspection

Inspect the generated documents and chunks across `data/processed/IL-TUR-PCR/`.

In [ ]:
processed_dir = project_root / "data" / "processed" / "IL-TUR-PCR"
splits = ["train_queries", "train_candidates", "dev_queries", "dev_candidates", "test_queries", "test_candidates"]

records = []
for s in splits:
    doc_p = processed_dir / s / "documents.parquet"
    chk_p = processed_dir / s / "chunks.parquet"
    if doc_p.exists() and chk_p.exists():
        d_table = pq.read_table(doc_p, columns=["id", "sentence_count", "word_count"])
        c_table = pq.read_table(chk_p, columns=["chunk_id", "token_count"])
        records.append({
            "Split": s,
            "Documents": len(d_table),
            "Total Chunks": len(c_table),
            "Avg Chunks/Doc": round(len(c_table) / len(d_table), 1),
            "Avg Words/Doc": round(float(pd.Series(d_table["word_count"]).mean()), 1)
        })

df_overview = pd.DataFrame(records)
display(df_overview) if 'display' in globals() else print(df_overview)

## 5. Summary Statistics and Distribution Visualizations

In [ ]:
stats_json = project_root / "results" / "tables" / "preprocessing_summary.json"
if stats_json.exists():
    with open(stats_json, "r", encoding="utf-8") as f:
        summary_data = json.load(f)
    print("Chunk Token Distribution:")
    print(json.dumps(summary_data["chunk_token_distribution"], indent=2))

fig_path = project_root / "results" / "figures" / "chunking_distributions.png"
if fig_path.exists():
    from IPython.display import Image, display
    display(Image(filename=str(fig_path)))